# Ingesta de relaciones película-género (incremental)

Lee `movie_genre.json` (JSON) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Carga la tabla Delta `movie_silver.movie_genre` con `MERGE` (llave `movie_id` + `genre_id` + `file_date`), particionada por `file_date`.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Lectura del archivo

Primero lee `movie_genre.json` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
movie_genre_df = (
    spark.read
        .format("json")
        .option("header", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_genre.json")
)

In [0]:
movie_genre_shema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("movieId", IntegerType(), True)
])
movie_genre_df = (
    spark.read
        .format("json")
        .schema(movie_genre_shema)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_genre.json")
)

### Transformación de columnas

Renombra `movieId` y `genreId` a snake_case y agrega `ingestion_date`, `env` y `file_date`.

In [0]:
movie_genre_df_final = (
    movie_genre_df
    .withColumnsRenamed({"genreId": "genre_id", "movieId": "movie_id"})
    .withColumns({"env": lit(v_env), "file_date": lit(v_file_date)})
)

movie_genre_df_final = add_ingestion_date(movie_genre_df_final)

### Escritura en Silver

Las celdas comentadas conservan estrategias anteriores (borrar partición + `append`). La carga activa es `merge_delta_lake` sobre `movie_silver.movie_genre` con la llave `movie_id` + `genre_id` + `file_date`.

In [0]:
#overwrite_partitioned_table("movie_silver", "movie_genre", "file_date", v_file_date)

In [0]:
"""
(
    movie_genre_df_final
    .write
        .mode("append")
        .partitionBy("file_date")
        .saveAsTable("movie_silver.movie_genre")
)
"""

In [0]:
merge_condition = 'trgt.movie_id = src.movie_id AND trgt.genre_id = src.genre_id AND trgt.file_date = src.file_date'
merge_delta_lake(input_df = movie_genre_df_final, schema_name = "movie_silver", table_name = "movie_genre", merge_condition = merge_condition, partition_column = "file_date")

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")

### Verificación

Cuenta registros por `file_date`. Está después de `dbutils.notebook.exit`, así que no se ejecuta cuando el orquestador invoca el notebook.

In [0]:
%sql
select file_date, COUNT(*) from movie_silver.movie_genre group by file_date